In [ ]:
import os
import asyncio
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
from dotenv import load_dotenv
from openai import AsyncOpenAI

In [ ]:
%pip install ragas

In [ ]:
from ragas.llms import llm_factory

# ── RAGAS ─────────────────────────────────────────────────────────────────────
from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings
from ragas import SingleTurnSample
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
    AnswerCorrectness,
)

# ── DeepEval ──────────────────────────────────────────────────────────────────
from deepeval.test_case import LLMTestCase, ToolCall
from deepeval.metrics import ToolCorrectnessMetric

In [15]:
from huggingface_hub import snapshot_download
HF_TOKEN = os.getenv("HF_TOKEN")
snapshot_download(
    repo_id="sentence-transformers/all-MiniLM-L6-v2", 
    local_dir="./all-MiniLM-L6-v2", 
    local_dir_use_symlinks=False,
    token=HF_TOKEN

)

RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6a3b425d-37ab165367ec754631ed9517;40809988-180a-4f8d-9474-37a3df931d85)

Repository Not Found for url: https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/revision/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
User Access Token "MyFavoriteToken" is expired

In [12]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
JUDGE_GROQ_API_KEY = os.getenv("JUDGE_GROQ", GROQ_API_KEY)
HF_TOKEN = os.getenv("HF_TOKEN")


groq_client = AsyncOpenAI(
    api_key=JUDGE_GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

judge_llm = llm_factory("llama-3.1-8b-instant", provider="openai", client=groq_client)

# ── Local Embeddings (zero API cost) ──────────────────────────────────────────
ragas_embeddings = HuggingFaceEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2", #384
    token=HF_TOKEN
)

print(f"✅ Judge LLM  : {type(judge_llm).__name__}")
print(f"✅ Embeddings : {type(ragas_embeddings).__name__} (local — no API key needed)")
using = "JUDGE_GROQ" if os.getenv("JUDGE_GROQ") else "GROQ_API_KEY (fallback)"
print(f"✅ Using key  : {using}")

OSError: sentence-transformers/all-MiniLM-L6-v2 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`